In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import RobertaTokenizer, RobertaForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset
from sklearn.metrics import accuracy_score
# 从 JSON 文件加载数据
df = pd.read_json(r'D:\PycharmProjects\TDFilter\experiment4\roberta_iterate\results.json')
print(df.head())
# 划分为训练集（20%）和剩余数据（80%）
train_data, remaining_data = train_test_split(df, test_size=0.7, random_state=42)

# 从剩余数据中划分测试集（5%）
test_size = 0.1025  # 5% of the original data corresponds to 5% of the remaining 80%
test_data, val_data = train_test_split(remaining_data, test_size=test_size, random_state=42)

# 显示结果
print(f"训练集大小: {len(train_data)}")
print(f"测试集大小: {len(test_data)}")
print(f"验证集大小: {len(val_data)}")

   flag  contradiction_score  confidence_score          topic  relation_score   
0     1                    0                 1   Paleobiology        0.842534  \
1     1                    1                 1       Genetics        0.929457   
2     1                    1                 1   Parasitology        0.830472   
3     1                    1                 1  Human Biology        0.805540   
4     1                    1                 1        Ecology        0.924844   

                                           knowledge   
0  Sedimentary rocks are typically dated using re...  \
1  The study of Mendelian genetics has led to adv...   
2  20. In the case of the nematode Wuchereria ban...   
3  Peripheral nerves are bundles of axons that ca...   
4  Conservation biology aims to protect and resto...   

                                   matchingKnowledge  
0  Fossils can be dated using techniques such as ...  
1  Mendel's work is a cornerstone of genetic educ...  
2  Certain 

In [38]:
# 定义数据集类
class KnowledgeDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __getitem__(self, idx):
        item = {key: val[idx].clone().detach() for key, val in self.encodings.items()}  # 使用 clone().detach()
        item['labels'] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

In [39]:
# 文本编码
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')

def encode_data(inputs):
    return tokenizer(
        inputs['knowledge'].tolist(),
        inputs['matchingKnowledge'].tolist(),
        padding=True,
        truncation=True,
        return_tensors='pt'
    )

# 编码数据集
train_encodings = encode_data(train_data)
val_encodings = encode_data(val_data)
test_encodings = encode_data(test_data)

In [28]:
# 创建数据集对象
train_dataset = KnowledgeDataset(train_encodings, train_data['flag'].tolist())
val_dataset = KnowledgeDataset(val_encodings, val_data['flag'].tolist())
test_dataset = KnowledgeDataset(test_encodings, test_data['flag'].tolist())

In [29]:
# 定义计算指标的函数
def compute_metrics(pred):
    labels = pred.label_ids
    preds = pred.predictions.argmax(-1)
    accuracy = accuracy_score(labels, preds)
    return {'accuracy': accuracy}

In [31]:
# 创建模型
model = RobertaForSequenceClassification.from_pretrained('roberta-base', num_labels=2)


Some weights of RobertaForSequenceClassification were not initialized from the model checkpoint at roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [32]:
import wandb
wandb.login()

wandb: WARNING Calling wandb.login() after wandb.init() has no effect.


True

In [33]:
# 定义训练参数
training_args = TrainingArguments(
    output_dir="contradiction_biological_roberta_trueEnviroment_28",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=5,
    weight_decay=0.01,
    eval_strategy="epoch",
    report_to="wandb",
)

# 创建 Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics,
)

In [34]:
trainer.train()

Epoch,Training Loss,Validation Loss


TrainOutput(global_step=850, training_loss=0.2155992193783031, metrics={'train_runtime': 93.2948, 'train_samples_per_second': 145.078, 'train_steps_per_second': 9.111, 'total_flos': 563394255621300.0, 'train_loss': 0.2155992193783031, 'epoch': 5.0})

In [35]:
trainer.evaluate(test_dataset)

{'eval_loss': 0.4774972200393677,
 'eval_accuracy': 0.9136045709782288,
 'eval_runtime': 15.5455,
 'eval_samples_per_second': 652.988,
 'eval_steps_per_second': 40.848,
 'epoch': 5.0}